In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [4]:
def add_user_message(messages, user_message):
    messages.append({"role": "user", "content": user_message})
    return messages

def add_assistant_message(messages, assistant_message):
    messages.append({"role": "assistant", "content": assistant_message})
    return messages

def chat(messages, system_prompt = None, temperature = 1.0):
    
    # base parameters
    parameters = {
        "model": model,
        "max_tokens": 600,
        "messages": messages,
        "temperature": temperature
    }
    
    if system_prompt:
        parameters["system_prompt"] = system_prompt

    response = client.messages.create(**parameters)

    assistant_message = response.content[0].text
    add_assistant_message(messages, assistant_message)
    return assistant_message

In [ ]:
messages = []
add_user_message(messages, "generate a 1 sentence of a fake database")
stream = client.messages.create(
    model = model,
    max_tokens = 600,
    messages = messages,
    stream = True
)

# every single time claude answers, it will send a stream of events containing its messages
# since the user will experience a delay between asking and receiving the answer, we can utilize stream to keep user engaged
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CcoHnaCXU1RCumSxrVVVc', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=17, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Fake Database Sentence\n\n"The `users_log` table contains 2', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='.

In [14]:
# a better way to access stream using client.messages.stream()
add_user_message(messages, "generat e a 1 sentence description of a fake database")
with client.messages.stream(
    model=model,
    max_tokens=600,
    messages=messages,
    temperature=0.5,
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass
    
    final_message = stream.get_final_message()

In [12]:
final_message.content[0].text

'# Fake Database Description\n\nThe "CloudDream_Users" database contains 2.4 million records of fictional customers with randomly generated names, email addresses, purchase histories, and preference profiles spanning from 2015 to 2024.'